In [1]:
%pip install pandas scikit-learn matplotlib seaborn

import pandas as pd
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

file_path = './shaped-coffee-bean-products.json'
with open(file_path, 'r') as f:
    data = json.load(f)

products = [item['product'] for item in data]
df = pd.json_normalize(
    products,
    sep='_',
    meta=[
        ['roaster', 'name'],
        ['roaster', 'country'],
        ['process', 'name'],
        ['process', 'tag']
    ],
    errors='ignore'
)

df.rename(columns={
    'roaster_name': 'roaster_name',
    'roaster_country': 'roaster_country',
    'process_name': 'process_name',
    'process_tag': 'process_tag'
}, inplace=True)

for col in ['roaster_name', 'roaster_country', 'process_name', 'process_tag', 'roastDegree', 'price']:
    df[col] = df[col].fillna('unknown')

df['price_numeric'] = pd.to_numeric(df['price'], errors='coerce')

df['price_numeric'] = df['price_numeric'].fillna(np.inf)

df['flavor_tags'] = df['flavors'].apply(lambda x: ', '.join([i['name'] for i in x]) if isinstance(x, list) else 'unknown')

df['producer_names'] = df['producers'].apply(lambda x: ', '.join([i['name'] for i in x]) if isinstance(x, list) else 'unknown')

df['combined_features'] = (
    df['roastDegree'].astype(str) + ' ' +
    df['process_name'].astype(str) + ' ' +
    df['process_tag'].astype(str) + ' ' +
    df['flavor_tags'] + ' ' +
    df['producer_names'] + ' ' +
    df['roaster_name'] + ' ' +
    df['roaster_country']
)



vectorizer = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')
tfidf_matrix = vectorizer.fit_transform(df['combined_features'])


def find_coffee(desired_roast=None, desired_country=None, desired_max_price=None,
                desired_process=None, desired_process_tag=None, desired_flavors=None,
                desired_producer=None, desired_roaster=None, top_n=5):
    query_parts = []
    if desired_roast:
        query_parts.append(str(desired_roast))
    if desired_process:
        query_parts.append(str(desired_process))
    if desired_process_tag:
        query_parts.append(str(desired_process_tag))
    if desired_flavors:
        query_parts.append(str(desired_flavors))
    if desired_producer:
        query_parts.append(str(desired_producer))
    if desired_roaster:
        query_parts.append(str(desired_roaster))
    if desired_country:
        query_parts.append(str(desired_country))

    query_string = ' '.join(query_parts)

    if not query_string:
        return "Please provide at least one desired attribute (roast, country, etc.)."

    query_vector = vectorizer.transform([query_string])
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    df['similarity'] = similarity_scores

    filtered_df = df.copy()
    if desired_max_price is not None:
        filtered_df = filtered_df[filtered_df['price_numeric'] <= desired_max_price]

    recommended_df = filtered_df.sort_values(by='similarity', ascending=False).head(top_n)

    if recommended_df.empty and desired_max_price is not None:
        print(f"⚠️ Warning: No coffees found under price ${desired_max_price}. Re-running without price filter.")
        recommended_df = df.sort_values(by='similarity', ascending=False).head(top_n)
        if recommended_df.empty:
            return "No coffees found matching the desired attributes."
        else:
            print("Showing most similar coffees regardless of price:")

    elif recommended_df.empty:
        return "No coffees found matching the desired attributes."

    return recommended_df[['name', 'roastDegree', 'roaster_country', 'price', 'flavor_tags', 'similarity']]


results = find_coffee(
    desired_roast="dark",
    desired_country="spain",
    desired_max_price=25,
    desired_flavors="chocolate, nutty"
)
display(results)

results_light_fruity = find_coffee(
    desired_roast="dark",
    desired_country="belgium",
    desired_flavors="chocolate, berry",
    desired_max_price=None
)
display(results_light_fruity)

results_light_fruity2 = find_coffee(
    desired_roast=None,
    desired_country="usa",
    desired_flavors="plum, berry",
    desired_max_price=None
)
display(results_light_fruity2)


Note: you may need to restart the kernel to use updated packages.


,name,roastDegree,roaster_country,price,flavor_tags,similarity
9898,antithesis,dark,usa,16.00,"dark chocolate, sweet cream, chocolate, caramel",0.428261
8522,dark roastbrazil,dark,germany,13.80,"chocolate, dark chocolate",0.420989
3766,crazy goat blend,medium dark,usa,16.95,"dark chocolate, toffee, chocolate, spice, caramel",0.402535
9790,forty-six,dark,usa,19.00,"chocolate, dark chocolate, sweet cream",0.402358
199,northwesterly blend,dark,usa,20.00,"spice, chocolate, dark chocolate, cream",0.402019


,name,roastDegree,roaster_country,price,flavor_tags,similarity
6098,"colombia, la esperanza, natural",omni,belgium,25.02,"dark chocolate, cran berry, prune, dried fruit...",0.487833
4403,zero waste blend,omni,belgium,6.25,,0.437599
10173,the traditional,dark,usa,18.00,"nut, dark chocolate, marzipan, berry, chocolat...",0.407308
1572,bulletin blend coffee,medium dark,usa,17.50,"dark chocolate, berry, chocolate, cherry, swee...",0.407128
9898,antithesis,dark,usa,16.00,"dark chocolate, sweet cream, chocolate, caramel",0.405436


,name,roastDegree,roaster_country,price,flavor_tags,similarity
9971,cambrrrr winter blend,unknown,usa,23.00,"plum, berry, rasp berry, sweet, stone fruit",0.430949
4501,colombia | rio bamisa | caturron | natural,omni,usa,28.00,"black berry, tropical fruit, plum, berry, melo...",0.411751
2156,sapsucker espresso,espresso,usa,26.00,"tangerine, tropical fruit, cran berry, plum, b...",0.401119
9764,"la aurora, costa rica",unknown,usa,21.00,"currant, plum, berry, tea, stone fruit",0.400869
4339,colombia | el indio | caturra | anaerobic natural,omni,usa,25.00,"black berry, plum, chocolate, berry, stone fruit",0.398894
